In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Set seeds
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU'))} GPU(s)")

TensorFlow version: 2.19.0
Keras version: 3.10.0
GPU Available: 0 GPU(s)


# CONFIG

In [5]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [6]:
BASE_DIR = "/content/drive/MyDrive/dataset_bing"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50

# Category mapping
CATEGORY_TO_ID = {
    'Amphibiens': 0,
    'Insectes': 1,
    'Mammiferes': 2,
    'Oiseaux': 3,
    'Poissons': 4,
    'Reptiles': 5
}

CATEGORY_ANIMALS = {
    'Amphibiens': ['Grenouille', 'Salamandre'],
    'Insectes': ['Abeille', 'Coccinelle', 'Fourmi', 'Papillon'],
    'Mammiferes': ['Chat', 'Cheval', 'Chien', 'Girafe', 'Lion', 'Ours', 'Singe', 'Tigre', 'Zèbre', 'Éléphant'],
    'Oiseaux': ['Aigle', 'Canard', 'Flamant rose', 'Hibou', 'Manchot', 'Perroquet'],
    'Poissons': ['Poisson rouge', 'Poisson-clown', 'Requin', 'Saumon'],
    'Reptiles': ['Crocodile', 'Lézard', 'Serpent', 'Tortue']
}

#ANALYSE DATASET

In [7]:
def analyze_dataset(base_dir):
    """Analyze dataset structure"""
    dataset_info = {}
    total_images = 0
    species_to_category = {}
    species_list = []

    for category in os.listdir(base_dir):
        category_path = os.path.join(base_dir, category)
        if os.path.isdir(category_path):
            dataset_info[category] = {}
            for animal in os.listdir(category_path):
                animal_path = os.path.join(category_path, animal)
                if os.path.isdir(animal_path):
                    count = len([f for f in os.listdir(animal_path)
                               if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
                    dataset_info[category][animal] = count
                    total_images += count
                    species_to_category[animal] = category
                    species_list.append(animal)

    return dataset_info, total_images, species_to_category, species_list

print("\n" + "="*60)
print("ANALYZING DATASET")
print("="*60)

dataset_info, total_images, species_to_category, species_list = analyze_dataset(BASE_DIR)

print(f"\nTotal Images: {total_images}")
print(f"Total Categories: {len(dataset_info)}")
print(f"Total Species: {len(species_list)}")

for category, animals in dataset_info.items():
    cat_total = sum(animals.values())
    print(f"\n{category}: {cat_total} images")
    for animal, count in animals.items():
        print(f"  - {animal}: {count} images")



ANALYZING DATASET

Total Images: 1474
Total Categories: 6
Total Species: 30

Oiseaux: 295 images
  - Perroquet: 49 images
  - Flamant rose: 49 images
  - Manchot: 50 images
  - Hibou: 48 images
  - Aigle: 49 images
  - Canard: 50 images

Poissons: 192 images
  - Saumon: 48 images
  - Poisson rouge: 47 images
  - Requin: 48 images
  - Poisson-clown: 49 images

Amphibiens: 97 images
  - Salamandre: 49 images
  - Grenouille: 48 images

Reptiles: 197 images
  - Serpent: 47 images
  - Crocodile: 50 images
  - Lézard: 50 images
  - Tortue: 50 images

Mammiferes: 493 images
  - Éléphant: 49 images
  - Zèbre: 50 images
  - Singe: 49 images
  - Tigre: 50 images
  - Lion: 49 images
  - Ours: 49 images
  - Girafe: 50 images
  - Chien: 51 images
  - Cheval: 50 images
  - Chat: 46 images

Insectes: 200 images
  - Fourmi: 50 images
  - Papillon: 50 images
  - Abeille: 50 images
  - Coccinelle: 50 images


# GEN DATA

In [8]:
class HierarchicalDataGenerator(keras.utils.Sequence):
    """Custom generator that returns both category and species labels"""

    def __init__(self, base_dir, species_to_category, img_size, batch_size,
                 augmentation=None, subset='training', validation_split=0.2, shuffle=True):
        self.base_dir = base_dir
        self.species_to_category = species_to_category
        self.img_size = img_size
        self.batch_size = batch_size
        self.shuffle = shuffle

        # Collect all image paths
        self.image_paths = []
        self.species_labels = []
        self.category_labels = []

        species_id = 0
        self.species_to_id = {}

        for category in os.listdir(base_dir):
            category_path = os.path.join(base_dir, category)
            if not os.path.isdir(category_path):
                continue

            category_id = CATEGORY_TO_ID.get(category, 0)

            for species in os.listdir(category_path):
                species_path = os.path.join(category_path, species)
                if not os.path.isdir(species_path):
                    continue

                if species not in self.species_to_id:
                    self.species_to_id[species] = species_id
                    species_id += 1

                for img_file in os.listdir(species_path):
                    if img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
                        img_path = os.path.join(species_path, img_file)
                        self.image_paths.append(img_path)
                        self.species_labels.append(self.species_to_id[species])
                        self.category_labels.append(category_id)

        # Split into train/validation
        n_samples = len(self.image_paths)
        indices = np.arange(n_samples)
        np.random.shuffle(indices)

        split_idx = int(n_samples * (1 - validation_split))

        if subset == 'training':
            indices = indices[:split_idx]
        else:
            indices = indices[split_idx:]

        self.image_paths = [self.image_paths[i] for i in indices]
        self.species_labels = [self.species_labels[i] for i in indices]
        self.category_labels = [self.category_labels[i] for i in indices]

        self.n_samples = len(self.image_paths)
        self.num_species = len(self.species_to_id)
        self.num_categories = len(CATEGORY_TO_ID)

        # Setup augmentation
        if augmentation:
            self.datagen = ImageDataGenerator(**augmentation)
        else:
            self.datagen = ImageDataGenerator(rescale=1./255)

        self.indexes = np.arange(self.n_samples)
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __len__(self):
        return int(np.ceil(self.n_samples / self.batch_size))

    def __getitem__(self, index):
        indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]

        batch_x = np.zeros((len(indexes), *self.img_size, 3), dtype=np.float32)
        batch_y_species = np.zeros((len(indexes), self.num_species), dtype=np.float32)
        batch_y_category = np.zeros((len(indexes), self.num_categories), dtype=np.float32)

        for i, idx in enumerate(indexes):
            img = keras.preprocessing.image.load_img(
                self.image_paths[idx],
                target_size=self.img_size
            )
            img_array = keras.preprocessing.image.img_to_array(img)
            img_array = self.datagen.random_transform(img_array)
            img_array = img_array / 255.0

            batch_x[i] = img_array
            batch_y_species[i, self.species_labels[idx]] = 1
            batch_y_category[i, self.category_labels[idx]] = 1

        return batch_x, {'species_output': batch_y_species, 'category_output': batch_y_category}

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

# MAPPINGS

In [9]:
print("\n" + "="*60)
print("CREATING DATA GENERATORS")
print("="*60)

train_augmentation = {
    'rotation_range': 20,
    'width_shift_range': 0.2,
    'height_shift_range': 0.2,
    'shear_range': 0.2,
    'zoom_range': 0.2,
    'horizontal_flip': True,
    'fill_mode': 'nearest'
}

train_generator = HierarchicalDataGenerator(
    BASE_DIR,
    species_to_category,
    IMG_SIZE,
    BATCH_SIZE,
    augmentation=train_augmentation,
    subset='training',
    validation_split=0.2,
    shuffle=True
)

val_generator = HierarchicalDataGenerator(
    BASE_DIR,
    species_to_category,
    IMG_SIZE,
    BATCH_SIZE,
    augmentation=None,
    subset='validation',
    validation_split=0.2,
    shuffle=False
)

print(f"Training samples: {train_generator.n_samples}")
print(f"Validation samples: {val_generator.n_samples}")
print(f"Number of species: {train_generator.num_species}")
print(f"Number of categories: {train_generator.num_categories}")



CREATING DATA GENERATORS
Training samples: 1179
Validation samples: 295
Number of species: 30
Number of categories: 6


# VIZ TEST

In [10]:
print("\n" + "="*60)
print("SAVING CLASS MAPPINGS")
print("="*60)

mappings = {
    'species_to_id': train_generator.species_to_id,
    'id_to_species': {str(v): k for k, v in train_generator.species_to_id.items()},
    'category_to_id': CATEGORY_TO_ID,
    'id_to_category': {str(v): k for k, v in CATEGORY_TO_ID.items()},
    'species_to_category': species_to_category
}

with open('class_mappings.json', 'w', encoding='utf-8') as f:
    json.dump(mappings, f, ensure_ascii=False, indent=2)

print("✓ Saved class mappings to 'class_mappings.json'")


SAVING CLASS MAPPINGS
✓ Saved class mappings to 'class_mappings.json'


# BUILD MODEL

In [11]:
print("\n" + "="*60)
print("GENERATING SAMPLE VISUALIZATIONS")
print("="*60)

def plot_samples(generator, num_samples=9):
    batch_x, batch_y = generator[0]

    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    axes = axes.ravel()

    # Get mappings from generator and CATEGORY_TO_ID
    id_to_species = {v: k for k, v in generator.species_to_id.items()}
    id_to_category = {v: k for k, v in CATEGORY_TO_ID.items()}

    for i in range(min(num_samples, len(batch_x))):
        axes[i].imshow(batch_x[i])

        species_idx = np.argmax(batch_y['species_output'][i])
        category_idx = np.argmax(batch_y['category_output'][i])

        species_name = id_to_species.get(species_idx, "Unknown")
        category_name = id_to_category.get(category_idx, "Unknown")

        axes[i].set_title(f"{species_name}\n({category_name})", fontsize=10)
        axes[i].axis('off')

    plt.tight_layout()
    plt.savefig('sample_training_images.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("✓ Saved: sample_training_images.png")

plot_samples(train_generator)



GENERATING SAMPLE VISUALIZATIONS
✓ Saved: sample_training_images.png


In [12]:
print("\n" + "="*60)
print("BUILDING HIERARCHICAL MODEL")
print("="*60)

def create_hierarchical_model(num_species, num_categories, img_size=IMG_SIZE):
    """
    Create a model with two outputs:
    1. Species classification (30 classes)
    2. Category classification (6 classes)
    """

    print("Loading MobileNetV2 base model...")
    # Load pre-trained MobileNetV2
    base_model = MobileNetV2(
        input_shape=(*img_size, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False
    print(f"✓ Base model loaded ({len(base_model.layers)} layers)")

    # Input layer
    inputs = keras.Input(shape=(*img_size, 3), name='image_input')

    # Base model
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)

    # Shared layers
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu', name='shared_dense_1')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    shared_features = layers.Dense(256, activation='relu', name='shared_dense_2')(x)

    # Species branch (detailed classification)
    species_x = layers.Dropout(0.2)(shared_features)
    species_output = layers.Dense(
        num_species,
        activation='softmax',
        name='species_output'
    )(species_x)

    # Category branch (coarse classification)
    category_x = layers.Dropout(0.2)(shared_features)
    category_output = layers.Dense(
        num_categories,
        activation='softmax',
        name='category_output'
    )(category_x)

    # Create model
    model = keras.Model(
        inputs=inputs,
        outputs={'species_output': species_output, 'category_output': category_output},
        name='hierarchical_animal_classifier'
    )

    print("✓ Model architecture created")
    return model, base_model

model, base_model = create_hierarchical_model(
    train_generator.num_species,
    train_generator.num_categories
)

# Compile model
print("Compiling model...")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss={
        'species_output': 'categorical_crossentropy',
        'category_output': 'categorical_crossentropy'
    },
    loss_weights={
        'species_output': 1.0,  # Main task
        'category_output': 0.5  # Auxiliary task (helps learning)
    },
    metrics={
        'species_output': ['accuracy'],
        'category_output': ['accuracy']
    }
)

print("✓ Model compiled successfully!")
print("\nModel Summary:")
model.summary()


BUILDING HIERARCHICAL MODEL
Loading MobileNetV2 base model...
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
✓ Base model loaded (154 layers)
✓ Model architecture created
Compiling model...
✓ Model compiled successfully!

Model Summary:


Model: "hierarchical_animal_classifier"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_1.00_2… │ (None, 7, 7,      │  2,257,984 │ image_input[0][0] │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ mobilenetv2_1.00… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 1280)      │      5,120 │ global_average_p… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 1280)      │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dense_1      │ (None, 512)       │    655,872 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512)       │      2,048 │ shared_dense_1[0… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 512)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dense_2      │ (None, 256)       │    131,328 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 256)       │          0 │ shared_dense_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 256)       │          0 │ shared_dense_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ category_output     │ (None, 6)         │      1,542 │ dropout_3[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ species_output      │ (None, 30)        │      7,710 │ dropout_2[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,061,604 (11.68 MB)

 Trainable params: 800,036 (3.05 MB)

 Non-trainable params: 2,261,568 (8.63 MB)

In [13]:
print("\n" + "="*60)
print("SETTING UP TRAINING CALLBACKS")
print("="*60)

callbacks = [
    EarlyStopping(
        monitor='val_species_output_accuracy',
        mode='max',  # We want to MAXIMIZE accuracy
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        mode='min',  # We want to MINIMIZE loss
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        'best_hierarchical_model.keras',
        monitor='val_species_output_accuracy',
        mode='max',  # Save the model with highest accuracy
        save_best_only=True,
        verbose=1
    )
]

print("✓ Callbacks configured:")
print("  - EarlyStopping (patience=10)")
print("  - ReduceLROnPlateau (patience=5)")
print("  - ModelCheckpoint")


SETTING UP TRAINING CALLBACKS
✓ Callbacks configured:
  - EarlyStopping (patience=10)
  - ReduceLROnPlateau (patience=5)
  - ModelCheckpoint


In [14]:
print("\n" + "="*60)
print("TRAINING MODEL - PHASE 1: Frozen Base")
print("="*60)
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Training samples: {train_generator.n_samples}")
print(f"Validation samples: {val_generator.n_samples}")
print("="*60 + "\n")

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print("\n✓ Phase 1 training complete!")


TRAINING MODEL - PHASE 1: Frozen Base
Epochs: 50
Batch size: 32
Training samples: 1179
Validation samples: 295

Epoch 1/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 14s/step - category_output_accuracy: 0.3773 - category_output_loss: 1.8470 - loss: 4.2764 - species_output_accuracy: 0.1607 - species_output_loss: 3.3533 
Epoch 1: val_species_output_accuracy improved from -inf to 0.81695, saving model to best_hierarchical_model.keras
37/37 ━━━━━━━━━━━━━━━━━━━━ 572s 15s/step - category_output_accuracy: 0.3811 - category_output_loss: 1.8340 - loss: 4.2513 - species_output_accuracy: 0.1647 - species_output_loss: 3.3347 - val_category_output_accuracy: 0.8644 - val_category_output_loss: 0.5708 - val_loss: 1.6225 - val_species_output_accuracy: 0.8169 - val_species_output_loss: 1.3612 - learning_rate: 0.0010
Epoch 2/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - category_output_accuracy: 0.8000 - category_output_loss: 0.6197 - loss: 1.4185 - species_output_accuracy: 0.7023 - species_output_loss: 1.1080
Epoch 2:

In [15]:
print("\n" + "="*60)
print("TRAINING MODEL - PHASE 2: Fine-tuning")
print("="*60)

# Unfreeze last 30 layers
base_model.trainable = True
frozen_layers = 0
for layer in base_model.layers[:-30]:
    layer.trainable = False
    frozen_layers += 1

print(f"Frozen layers: {frozen_layers}")
print(f"Trainable layers: {len(base_model.layers) - frozen_layers}")

# Recompile with lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss={
        'species_output': 'categorical_crossentropy',
        'category_output': 'categorical_crossentropy'
    },
    loss_weights={
        'species_output': 1.0,
        'category_output': 0.5
    },
    metrics={
        'species_output': ['accuracy'],
        'category_output': ['accuracy']
    }
)

print("✓ Model recompiled with learning_rate=1e-5")
print("="*60 + "\n")

history_fine = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print("\n✓ Phase 2 fine-tuning complete!")


TRAINING MODEL - PHASE 2: Fine-tuning
Frozen layers: 124
Trainable layers: 30
✓ Model recompiled with learning_rate=1e-5

Epoch 1/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - category_output_accuracy: 0.9075 - category_output_loss: 0.3152 - loss: 0.5540 - species_output_accuracy: 0.8817 - species_output_loss: 0.3963
Epoch 1: val_species_output_accuracy improved from 0.96949 to 0.97288, saving model to best_hierarchical_model.keras
37/37 ━━━━━━━━━━━━━━━━━━━━ 144s 4s/step - category_output_accuracy: 0.9074 - category_output_loss: 0.3148 - loss: 0.5535 - species_output_accuracy: 0.8816 - species_output_loss: 0.3961 - val_category_output_accuracy: 0.9763 - val_category_output_loss: 0.1560 - val_loss: 0.2416 - val_species_output_accuracy: 0.9729 - val_species_output_loss: 0.3224 - learning_rate: 1.0000e-05
Epoch 2/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - category_output_accuracy: 0.9218 - category_output_loss: 0.2596 - loss: 0.4882 - species_output_accuracy: 0.8784 - species_output_loss: 0

In [16]:
print("\n" + "="*60)
print("GENERATING TRAINING VISUALIZATIONS")
print("="*60)

def plot_training_history(history, history_fine):
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    # Combine histories
    metrics = {
        'species_output_accuracy': [],
        'val_species_output_accuracy': [],
        'category_output_accuracy': [],
        'val_category_output_accuracy': []
    }

    for key in metrics.keys():
        if key in history.history:
            metrics[key].extend(history.history[key])
        if key in history_fine.history:
            metrics[key].extend(history_fine.history[key])

    epochs = range(len(metrics['species_output_accuracy']))

    # Species accuracy
    axes[0, 0].plot(epochs, metrics['species_output_accuracy'], label='Train', linewidth=2)
    axes[0, 0].plot(epochs, metrics['val_species_output_accuracy'], label='Validation', linewidth=2)
    axes[0, 0].set_title('Species Classification Accuracy', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Category accuracy
    axes[0, 1].plot(epochs, metrics['category_output_accuracy'], label='Train', linewidth=2)
    axes[0, 1].plot(epochs, metrics['val_category_output_accuracy'], label='Validation', linewidth=2)
    axes[0, 1].set_title('Category Classification Accuracy', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # Loss
    loss_keys = ['loss', 'val_loss']
    for key in loss_keys:
        loss_data = []
        if key in history.history:
            loss_data.extend(history.history[key])
        if key in history_fine.history:
            loss_data.extend(history_fine.history[key])
        axes[1, 0].plot(epochs, loss_data, label=key.replace('_', ' ').title(), linewidth=2)

    axes[1, 0].set_title('Total Loss', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Loss')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Final accuracies comparison
    final_species_acc = metrics['val_species_output_accuracy'][-1]
    final_category_acc = metrics['val_category_output_accuracy'][-1]

    colors = ['#667eea', '#764ba2']
    bars = axes[1, 1].bar(['Species\n(30 classes)', 'Category\n(6 classes)'],
                          [final_species_acc, final_category_acc],
                          color=colors)
    axes[1, 1].set_title('Final Validation Accuracy', fontsize=14, fontweight='bold')
    axes[1, 1].set_ylabel('Accuracy')
    axes[1, 1].set_ylim([0, 1])
    axes[1, 1].grid(True, alpha=0.3, axis='y')

    for i, (v, bar) in enumerate(zip([final_species_acc, final_category_acc], bars)):
        axes[1, 1].text(bar.get_x() + bar.get_width()/2, v + 0.02,
                       f'{v:.2%}', ha='center', fontweight='bold', fontsize=12)

    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("✓ Saved: training_history.png")

plot_training_history(history, history_fine)


GENERATING TRAINING VISUALIZATIONS
✓ Saved: training_history.png


In [17]:
print("\n" + "="*60)
print("EVALUATING MODEL")
print("="*60)

# Load best model
model = keras.models.load_model('best_hierarchical_model.keras')

# Evaluate
results = model.evaluate(val_generator, verbose=1)

print("\n" + "="*60)
print("FINAL EVALUATION RESULTS")
print("="*60)
for i, metric_name in enumerate(model.metrics_names):
    value = results[i]
    if 'accuracy' in metric_name:
        print(f"{metric_name}: {value:.2%}")
    else:
        print(f"{metric_name}: {value:.4f}")

# ============================================================
# STEP 12: SAVE FINAL MODEL
# ============================================================
print("\n" + "="*60)
print("SAVING FINAL MODEL")
print("="*60)

model.save('hierarchical_animal_classifier.keras')
print("✓ Saved: hierarchical_animal_classifier.keras")

# Try to download files (for Google Colab)
try:
    from google.colab import files
    print("\nDownloading files...")
    files.download('hierarchical_animal_classifier.keras')
    files.download('class_mappings.json')
    files.download('training_history.png')
    files.download('sample_training_images.png')
    print("✓ Files downloaded!")
except:
    print("\n(Not in Colab - files saved locally)")



EVALUATING MODEL
10/10 ━━━━━━━━━━━━━━━━━━━━ 21s 2s/step - category_output_accuracy: 0.9754 - category_output_loss: 0.0778 - loss: 0.2248 - species_output_accuracy: 0.9738 - species_output_loss: 0.2148

FINAL EVALUATION RESULTS
loss: 0.2416
compile_metrics: 0.3224
species_output_loss: 0.1560
category_output_loss: 0.9763

SAVING FINAL MODEL
✓ Saved: hierarchical_animal_classifier.keras



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Files downloaded!


In [18]:
print("\n" + "="*60)
print("🎉 TRAINING COMPLETE!")
print("="*60)
print("\nFiles generated:")
print("  ✓ hierarchical_animal_classifier.keras - Final model")
print("  ✓ best_hierarchical_model.keras - Best model checkpoint")
print("  ✓ class_mappings.json - All class mappings")
print("  ✓ training_history.png - Training curves")
print("  ✓ sample_training_images.png - Sample images")
print("\n" + "="*60)
print("\nNext steps:")
print("1. Download the .keras and .json files")
print("2. Copy them to your PC project folder")
print("3. Run the Flask web app!")
print("="*60)


🎉 TRAINING COMPLETE!

Files generated:
  ✓ hierarchical_animal_classifier.keras - Final model
  ✓ best_hierarchical_model.keras - Best model checkpoint
  ✓ class_mappings.json - All class mappings
  ✓ training_history.png - Training curves
  ✓ sample_training_images.png - Sample images


Next steps:
1. Download the .keras and .json files
2. Copy them to your PC project folder
3. Run the Flask web app!
